In [2]:
                                     #"Cat Dog classification CNN from Scratch & Transfert Learning"
#Montage Google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#ibliothèques
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

In [4]:
# GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Utilisation de : {device}")

🔥 Utilisation de : cuda


In [5]:
#Fixons la graine de reproductibilité
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)


In [7]:
#chemin dossier data sur drive
data_dir = '/content/drive/MyDrive/cnn_cat_dog_image_classification-20260530T210257Z-3-001/cnn_cat_dog_image_classification/Cat_Dog_data'

In [8]:
#Transformations et chargement des data
# Transformations
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(), #data augmentation pour l'entraînement
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Pour le test : pas d'augmentation, seulement redimensionnement et crop central
test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



In [9]:
# Chargement des datasets
train_data = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transforms)
test_data = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=test_transforms)

print(f"Classes : {train_data.class_to_idx}")  # devrait donner {'cat':0, 'dog':1}
print(f"Taille train : {len(train_data)} images")
print(f"Taille test  : {len(test_data)} images")

Classes : {'cat': 0, 'dog': 1}
Taille train : 22595 images
Taille test  : 2512 images


In [10]:
#extraction de  20% du train set pour la validation (surveillance pendant l'entraînement) "split"
val_ratio = 0.2
val_size = int(len(train_data) * val_ratio)
train_size = len(train_data) - val_size
train_subset, val_subset = random_split(train_data, [train_size, val_size])


In [12]:
# Dataloaders
batch_size = 32
trainloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2) #pas de melange
testloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2) #pas de melange

print(f"Train set (après split) : {train_size} images")
print(f"Validation set : {val_size} images")
print(f"Test set : {len(test_data)} images")
print(f"train set : {len(train_data)} images")


Train set (après split) : 18076 images
Validation set : 4519 images
Test set : 2512 images
train set : 22595 images


In [17]:
#definitions des modèles CNN
#1. CNN from Scratch avec 4 blocs convolutifs + BatchNorm + Dropout
class CNNFromScratch(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Bloc 1 : 3 -> 32
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout2d(0.25)

        # Bloc 2 : 32 -> 64
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout2d(0.25)

        # Bloc 3 : 64 -> 128
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.drop3 = nn.Dropout2d(0.25)

        # Bloc 4 : 128 -> 256
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(2)
        self.drop4 = nn.Dropout2d(0.25)

        # Classifieur
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.pool1(self.drop1(torch.relu(self.bn1(self.conv1(x)))))
        x = self.pool2(self.drop2(torch.relu(self.bn2(self.conv2(x)))))
        x = self.pool3(self.drop3(torch.relu(self.bn3(self.conv3(x)))))
        x = self.pool4(self.drop4(torch.relu(self.bn4(self.conv4(x)))))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [18]:
#2.Modèle du transfert learning avec ResNet18 pré-entraîné
def get_resnet18_transfer(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    # Gelons toutes les couches sauf la dernière et layer4
    for param in model.parameters():
        param.requires_grad = False
    # Dégeler layer4 pour un fine-tuning plus précis
    for param in model.layer4.parameters():
        param.requires_grad = True
    # Remplacer la tête de classification
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes)
    )
    return model

In [19]:
#3. Fonctions d'entrainement et de validation
def train_one_epoch(model, loader, criterion, optimizer, device, writer, epoch):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    loop = tqdm(loader, desc=f"Train Epoch {epoch+1}", leave=False)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_prec = precision_score(all_labels, all_preds, average='binary')
    epoch_rec = recall_score(all_labels, all_preds, average='binary')
    writer.add_scalar('Train/Loss', epoch_loss, epoch)
    writer.add_scalar('Train/Acc', epoch_acc, epoch)
    writer.add_scalar('Train/Prec', epoch_prec, epoch)
    writer.add_scalar('Train/Rec', epoch_rec, epoch)
    return epoch_loss, epoch_acc, epoch_prec, epoch_rec

def validate(model, loader, criterion, device, writer, epoch):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_prec = precision_score(all_labels, all_preds, average='binary')
    epoch_rec = recall_score(all_labels, all_preds, average='binary')
    writer.add_scalar('Val/Loss', epoch_loss, epoch)
    writer.add_scalar('Val/Acc', epoch_acc, epoch)
    writer.add_scalar('Val/Prec', epoch_prec, epoch)
    writer.add_scalar('Val/Rec', epoch_rec, epoch)
    return epoch_loss, epoch_acc, epoch_prec, epoch_rec, all_labels, all_preds

def train_model(model, model_name, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=15):
    best_val_acc = 0.0
    best_model_path = f"best_{model_name}.pth"
    history = {'train_loss':[],'train_acc':[],'train_prec':[],'train_rec':[],
               'val_loss':[],'val_acc':[],'val_prec':[],'val_rec':[]}
    writer = SummaryWriter(f"runs/{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        train_loss, train_acc, train_prec, train_rec = train_one_epoch(
            model, train_loader, criterion, optimizer, device, writer, epoch)
        val_loss, val_acc, val_prec, val_rec, _, _ = validate(
            model, val_loader, criterion, device, writer, epoch)

        for key, val in zip(['train_loss','train_acc','train_prec','train_rec'],
                            [train_loss, train_acc, train_prec, train_rec]):
            history[key].append(val)
        for key, val in zip(['val_loss','val_acc','val_prec','val_rec'],
                            [val_loss, val_acc, val_prec, val_rec]):
            history[key].append(val)

        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Prec: {train_prec:.4f}, Rec: {train_rec:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}")

        scheduler.step()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"Meilleur modèle sauvegardé (val_acc={best_val_acc:.4f})")
    writer.close()
    # Recharger le meilleur modèle
    model.load_state_dict(torch.load(best_model_path))
    return model, history, best_val_acc

In [20]:
#4. Expérience A : CNN from Strach
print("\n" + "="*60)
print(" Expérience A : CNN FROM SCRATCH")
print("="*60)

model_scratch = CNNFromScratch().to(device)
criterion = nn.CrossEntropyLoss()

# Test de deux optimiseurs
optimizers_scratch = {
    'Adam': optim.Adam(model_scratch.parameters(), lr=0.001),
    'SGD': optim.SGD(model_scratch.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
}
best_scratch_acc = 0
best_scratch_opt = None
results_scratch = {}

for opt_name, optimizer in optimizers_scratch.items():
    print(f"\n Optimiseur : {opt_name}")
    model = CNNFromScratch().to(device)
    scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    model, history, best_acc = train_model(
        model, f"scratch_{opt_name}", trainloader, valloader,
        criterion, optimizer, scheduler, device, num_epochs=15
    )
    results_scratch[opt_name] = (model, history, best_acc)
    if best_acc > best_scratch_acc:
        best_scratch_acc = best_acc
        best_scratch_opt = opt_name

print(f"\n Meilleur optimiseur pour scratch : {best_scratch_opt} (val_acc={best_scratch_acc:.4f})")
model_scratch_best, hist_scratch_best, _ = results_scratch[best_scratch_opt]


 Expérience A : CNN FROM SCRATCH

 Optimiseur : Adam

Epoch 1/15


Train - Loss: 0.7178, Acc: 0.5056, Prec: 0.5091, Rec: 0.3660
Val   - Loss: 0.6959, Acc: 0.4983, Prec: 0.4130, Rec: 0.0084
Meilleur modèle sauvegardé (val_acc=0.4983)

Epoch 2/15


Train - Loss: 0.7177, Acc: 0.4998, Prec: 0.5010, Rec: 0.3552
Val   - Loss: 0.6962, Acc: 0.4997, Prec: 0.4815, Rec: 0.0115
Meilleur modèle sauvegardé (val_acc=0.4997)

Epoch 3/15


Train - Loss: 0.7191, Acc: 0.5053, Prec: 0.5088, Rec: 0.3567
Val   - Loss: 0.6954, Acc: 0.5006, Prec: 0.5143, Rec: 0.0159
Meilleur modèle sauvegardé (val_acc=0.5006)

Epoch 4/15


Train - Loss: 0.7232, Acc: 0.4954, Prec: 0.4948, Rec: 0.3509
Val   - Loss: 0.6962, Acc: 0.4979, Prec: 0.4138, Rec: 0.0106

Epoch 5/15


...

Train - Loss: 0.7321, Acc: 0.5017, Prec: 0.5014, Rec: 0.9089
Val   - Loss: 0.6956, Acc: 0.4999, Prec: 0.4999, Rec: 1.0000

Epoch 12/15


Train - Loss: 0.7317, Acc: 0.4991, Prec: 0.5000, Rec: 0.9028
Val   - Loss: 0.6963, Acc: 0.4999, Prec: 0.4999, Rec: 1.0000

Epoch 13/15


Train - Loss: 0.7319, Acc: 0.4978, Prec: 0.4993, Rec: 0.9004
Val   - Loss: 0.6959, Acc: 0.4999, Prec: 0.4999, Rec: 1.0000

Epoch 14/15


Train - Loss: 0.7313, Acc: 0.5004, Prec: 0.5007, Rec: 0.9055
Val   - Loss: 0.6958, Acc: 0.4999, Prec: 0.4999, Rec: 1.0000

Epoch 15/15


Train - Loss: 0.7320, Acc: 0.5003, Prec: 0.5006, Rec: 0.9065
Val   - Loss: 0.6955, Acc: 0.4999, Prec: 0.4999, Rec: 1.0000

 Meilleur optimiseur pour scratch : Adam (val_acc=0.5006)


In [ ]:
#5. Expérience B: Learning transfert
print("\n" + "="*60)
print(" Expérience B : Transfert learning avec (ResNet18)")
print("="*60)

optimizers_transfer = {
    'Adam': optim.Adam(get_resnet18_transfer().parameters(), lr=0.0001),
    'SGD': optim.SGD(get_resnet18_transfer().parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
}
best_transfer_acc = 0
best_transfer_opt = None
results_transfer = {}

for opt_name, optimizer in optimizers_transfer.items():
    print(f"\n Optimiseur : {opt_name}")
    model = get_resnet18_transfer().to(device)
    scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    model, history, best_acc = train_model(
        model, f"transfer_{opt_name}", trainloader, valloader,
        criterion, optimizer, scheduler, device, num_epochs=15
    )
    results_transfer[opt_name] = (model, history, best_acc)
    if best_acc > best_transfer_acc:
        best_transfer_acc = best_acc
        best_transfer_opt = opt_name

print(f"\n Meilleur optimiseur pour transfer learning : {best_transfer_opt} (val_acc={best_transfer_acc:.4f})")
model_transfer_best, hist_transfer_best, _ = results_transfer[best_transfer_opt]



 Expérience B : Transfert learning avec (ResNet18)

 Optimiseur : Adam

Epoch 1/15


Train - Loss: 0.7168, Acc: 0.5014, Prec: 0.5015, Rec: 0.7512
Val   - Loss: 0.7065, Acc: 0.5012, Prec: 0.5006, Rec: 0.8946
Meilleur modèle sauvegardé (val_acc=0.5012)

Epoch 2/15


Train - Loss: 0.7172, Acc: 0.5005, Prec: 0.5009, Rec: 0.7510
Val   - Loss: 0.7038, Acc: 0.4999, Prec: 0.4999, Rec: 0.8800

Epoch 3/15


Train - Loss: 0.7146, Acc: 0.5073, Prec: 0.5054, Rec: 0.7580
Val   - Loss: 0.7054, Acc: 0.5059, Prec: 0.5032, Rec: 0.9044
Meilleur modèle sauvegardé (val_acc=0.5059)

Epoch 4/15


Train - Loss: 0.7161, Acc: 0.5012, Prec: 0.5014, Rec: 0.7567
Val   - Loss: 0.7053, Acc: 0.5037, Prec: 0.5020, Rec: 0.8893

Epoch 5/15


...

Train - Loss: 0.7366, Acc: 0.4782, Prec: 0.4610, Rec: 0.2471
Val   - Loss: 0.7238, Acc: 0.4747, Prec: 0.4109, Rec: 0.1173

Epoch 11/15


Train - Loss: 0.7346, Acc: 0.4816, Prec: 0.4680, Rec: 0.2561
Val   - Loss: 0.7205, Acc: 0.4875, Prec: 0.4530, Rec: 0.1217

Epoch 12/15


Train - Loss: 0.7327, Acc: 0.4806, Prec: 0.4653, Rec: 0.2478
Val   - Loss: 0.7231, Acc: 0.4764, Prec: 0.4160, Rec: 0.1173

Epoch 13/15


Train - Loss: 0.7349, Acc: 0.4819, Prec: 0.4678, Rec: 0.2497
Val   - Loss: 0.7206, Acc: 0.4902, Prec: 0.4623, Rec: 0.1222
Meilleur modèle sauvegardé (val_acc=0.4902)

Epoch 14/15


Train - Loss: 0.7312, Acc: 0.4877, Prec: 0.4792, Rec: 0.2622
Val   - Loss: 0.7225, Acc: 0.4842, Prec: 0.4362, Rec: 0.1089

Epoch 15/15


In [ ]:
#6. Comparaison des courbes
plt.figure(figsize=(14,10))
plt.subplot(2,2,1)
plt.plot(hist_scratch_best['train_loss'], label='Scratch Train')
plt.plot(hist_scratch_best['val_loss'], label='Scratch Val')
plt.plot(hist_transfer_best['train_loss'], '--', label='Transfer Train')
plt.plot(hist_transfer_best['val_loss'], '--', label='Transfer Val')
plt.title('Loss'), plt.xlabel('Epoch'), plt.legend(), plt.grid()

plt.subplot(2,2,2)
plt.plot(hist_scratch_best['train_acc'], label='Scratch Train')
plt.plot(hist_scratch_best['val_acc'], label='Scratch Val')
plt.plot(hist_transfer_best['train_acc'], '--', label='Transfer Train')
plt.plot(hist_transfer_best['val_acc'], '--', label='Transfer Val')
plt.title('Accuracy'), plt.xlabel('Epoch'), plt.legend(), plt.grid()

plt.subplot(2,2,3)
plt.plot(hist_scratch_best['train_prec'], label='Scratch Train')
plt.plot(hist_scratch_best['val_prec'], label='Scratch Val')
plt.plot(hist_transfer_best['train_prec'], '--', label='Transfer Train')
plt.plot(hist_transfer_best['val_prec'], '--', label='Transfer Val')
plt.title('Precision'), plt.xlabel('Epoch'), plt.legend(), plt.grid()

plt.subplot(2,2,4)
plt.plot(hist_scratch_best['train_rec'], label='Scratch Train')
plt.plot(hist_scratch_best['val_rec'], label='Scratch Val')
plt.plot(hist_transfer_best['train_rec'], '--', label='Transfer Train')
plt.plot(hist_transfer_best['val_rec'], '--', label='Transfer Val')
plt.title('Recall'), plt.xlabel('Epoch'), plt.legend(), plt.grid()

plt.suptitle('Comparaison CNN From Scratch vs Transfer Learning')
plt.tight_layout()
plt.savefig('comparison_curves.png')
plt.show()

In [ ]:
#7. évaluation finale sur les données test

def final_evaluation(model, loader, device, model_name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='binary')
    rec = recall_score(all_labels, all_preds, average='binary')
    cm = confusion_matrix(all_labels, all_preds)
    print(f"\n {model_name} sur le test set :")
    print(f"   Accuracy  : {acc:.4f}")
    print(f"   Precision : {prec:.4f}")
    print(f"   Recall    : {rec:.4f}")
    return acc, prec, rec, cm, all_labels, all_preds

print("\n" + "="*60)
print(" ÉVALUATION SUR LE TEST SET (Data never view)")
print("="*60)

acc_scratch, prec_scratch, rec_scratch, cm_scratch, _, _ = final_evaluation(
    model_scratch_best, testloader, device, "CNN From Scratch")
acc_transfer, prec_transfer, rec_transfer, cm_transfer, _, _ = final_evaluation(
    model_transfer_best, testloader, device, "Transfer Learning (ResNet18)")

In [ ]:
#8. Matrices de confusion
fig, axes = plt.subplots(1,2, figsize=(12,5))
ConfusionMatrixDisplay(cm_scratch, display_labels=['Cat','Dog']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title(f'CNN From Scratch (test acc={acc_scratch:.4f})')
ConfusionMatrixDisplay(cm_transfer, display_labels=['Cat','Dog']).plot(ax=axes[1], cmap='Greens')
axes[1].set_title(f'Transfer Learning (test acc={acc_transfer:.4f})')
plt.tight_layout()
plt.savefig('confusion_matrices.png')
plt.show()

In [ ]:
#10. Analyse des erreurs (exemples sur test)
def show_misclassifications(model, loader, device, model_name, num=6):
    model.eval()
    misclassified = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            for i in range(len(images)):
                if preds[i] != labels[i] and len(misclassified) < num:
                    misclassified.append((images[i].cpu(), labels[i].item(), preds[i].item()))
            if len(misclassified) >= num:
                break
    # Dénormalisation
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    fig, axes = plt.subplots(2,3, figsize=(15,8))
    axes = axes.flatten()
    for i, (img, true, pred) in enumerate(misclassified):
        img = img * std + mean
        img = torch.clamp(img, 0, 1)
        axes[i].imshow(img.permute(1,2,0))
        axes[i].set_title(f"True: {train_data.classes[true]} | Pred: {train_data.classes[pred]}")
        axes[i].axis('off')
    for j in range(len(misclassified), len(axes)):
        axes[j].axis('off')
    plt.suptitle(f"Erreurs typiques - {model_name}")
    plt.tight_layout()
    plt.show()

print("\n Exemples de mauvaises classifications sur le test set")
show_misclassifications(model_scratch_best, testloader, device, "CNN From Scratch", num=6)
show_misclassifications(model_transfer_best, testloader, device, "Transfer Learning", num=6)

In [ ]:
# 11. Resumé final et sauvegarde des modéles finaux sur le drive
import pandas as pd
summary = pd.DataFrame({
    'Modèle': ['CNN From Scratch', 'Transfer Learning (ResNet18)'],
    'Accuracy (test)': [acc_scratch, acc_transfer],
    'Precision (test)': [prec_scratch, prec_transfer],
    'Recall (test)': [rec_scratch, rec_transfer],
    'Meilleur optimiseur (validation)': [best_scratch_opt, best_transfer_opt]
})
print("\n" + "="*60)
print("RÉSUMÉ FINAL DES PERFORMANCES")
print("="*60)
print(summary.to_string(index=False))

# Sauvegarde des modèles finaux sur Drive
torch.save(model_scratch_best.state_dict(), '/content/drive/MyDrive/cnn_scratch_final.pth')
torch.save(model_transfer_best.state_dict(), '/content/drive/MyDrive/resnet18_transfer_final.pth')
print("\n Modèles finaux sauvegardés sur votre Drive :")
print("   - cnn_scratch_final.pth")
print("   - resnet18_transfer_final.pth")